### 2025 한국 마사회 정보

In [90]:
import pandas as pd

df = pd.read_csv("C:/Users/Admin/Desktop/hhh/Horse/data_raw/raw_race_2025.csv", encoding="utf-8-sig")

In [91]:
df.head()

,chaksun,diffTot,divide,hrName,hrno,jkName,jkNo,meet,noracefl,prow,...,rcPlansu,rcRank,rcSex,rcSpcbu,rcTime,rcVtdusu,rundayth,track,weath,wgHr
0,24750000,0.000,0,마이티러브,50800,김태희,080603,서울,정상,113005.0,...,12,6,오픈,1,76.3,12,1,건조,맑음,407
1,0,34.125,0,지나송,51231,문정균,080337,서울,정상,10578.0,...,12,6,오픈,1,82.3,12,1,건조,맑음,503
2,0,12.125,0,위버멘쉬,51823,오수철,080608,서울,정상,105212.0,...,12,6,오픈,1,78.3,12,1,건조,맑음,454
3,6300000,1.625,0,과천퀸,52479,마이아,080616,서울,정상,124010.0,...,12,6,오픈,1,76.6,12,1,건조,맑음,455
4,0,13.125,0,또순이서울,51423,이동하,080562,서울,정상,215035.0,...,12,6,오픈,1,78.5,12,1,건조,맑음,456


In [92]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 11020 entries, 0 to 11019
Data columns (total 56 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   chaksun   11020 non-null  int64  
 1   diffTot   11020 non-null  float64
 2   divide    11020 non-null  int64  
 3   hrName    11020 non-null  str    
 4   hrno      11020 non-null  int64  
 5   jkName    11020 non-null  str    
 6   jkNo      11020 non-null  str    
 7   meet      11020 non-null  str    
 8   noracefl  11020 non-null  str    
 9   prow      11017 non-null  float64
 10  prowName  11017 non-null  str    
 11  prtr      11016 non-null  float64
 12  prtrName  11016 non-null  str    
 13  rankKind  11020 non-null  int64  
 14  rc10dusu  11020 non-null  int64  
 15  rcAge     10855 non-null  str    
 16  rcBudam   11020 non-null  int64  
 17  rcChul    11020 non-null  int64  
 18  rcCode    11020 non-null  str    
 19  rcDate    11020 non-null  int64  
 20  rcDiff2   11020 non-null  float64
 21  

In [93]:
df.isnull().sum()[df.isnull().sum() > 0]
df[df['wgHr'] == 0].index

Index([  170,   394,   551,   627,   703,   832,   892,  1203,  1292,  1314,
       ...
       10453, 10550, 10645, 10687, 10708, 10838, 10903, 10907, 10958, 10976],
      dtype='int64', length=115)

In [94]:
df[df['prow'].isnull()].index

Index([892, 2266, 10708], dtype='int64')

In [95]:
df[df == 0].index

RangeIndex(start=0, stop=11020, step=1)

In [96]:
(df['wgHr'] == 0).sum()

np.int64(115)

In [97]:
df['hrName'].nunique()

1973

In [98]:
def fill_weight(data: list[dict]) -> list[dict]:
    """
    경주마 몸무게 결측치 보정 함수
    """

    # 1) 전체 평균 계산용
    total_sum = 0
    total_cnt = 0

    # 2) 말(hrno)별 평균 계산용
    horse_sum = {}
    horse_cnt = {}

    # -------------------------
    # 1차 패스: 통계 계산
    # -------------------------
    for row in data:
        hrno = row["hrno"]
        w = row.get("wgHr")

        if w is None or w == 0:
            continue

        total_sum += w
        total_cnt += 1

        if hrno not in horse_sum:
            horse_sum[hrno] = 0
            horse_cnt[hrno] = 0

        horse_sum[hrno] += w
        horse_cnt[hrno] += 1

    # 평균 계산
    global_mean = total_sum / total_cnt if total_cnt > 0 else 0

    horse_mean = {}
    for h in horse_sum:
        horse_mean[h] = horse_sum[h] / horse_cnt[h]

    # -------------------------
    # 2차 패스: 결측치 채우기
    # -------------------------
    result = []

    for row in data:
        hrno = row["hrno"]
        w = row.get("wgHr")
        is_first = row.get("is_first_run", False)

        # 정상값 유지
        if w is not None and w != 0:
            result.append({
                "hrno": hrno,
                "wgHr": float(w)
            })
            continue

        # 결측 처리
        if is_first:
            fill_value = global_mean
        else:
            fill_value = horse_mean.get(hrno, global_mean)

        result.append({
            "hrno": hrno,
            "wgHr": float(fill_value)
        })

    return result

### 결측치 처리
- 말 무게 처리 (0인경우 - 마번이 같은 경우: 해당 말의 시즌 평균 몸무게
                     - 마번이 없는 경우: 평균 말의 몸무게)

####
- prow: 3개
- prowName: 3개
- prtr: 4개
### 데이터 없애도 되는 것
meet: 전부 서울
noracefl: 전부 정상

In [57]:
df['wgHr'] = df['wgHr'].astype(float).mask(
    df['wgHr'] == 0,
    df.groupby('hrno')['wgHr'].transform('mean')
)

In [62]:
print((df['wgHr'] == 0).sum())
df['wgHr'].isnull().sum()

10


np.int64(0)

In [63]:
import numpy as np
df['wgHr'] = df['wgHr'].astype(float)

# 1. 0 → NaN 처리
df['wgHr'] = df['wgHr'].replace(0, np.nan)

# 2. hrno 평균으로 채우기
df['wgHr'] = df['wgHr'].fillna(df.groupby('hrno')['wgHr'].transform('mean'))

# 3. 신규 말 처리
df['wgHr'] = df['wgHr'].fillna(df['wgHr'].mean())

In [ ]:
def fill_weight(data: list[dict]) -> list[dict]:
    """
    말 몸무게 결측치를 보정하는 함수

    INPUT
    -----
    data : list[dict]
        각 원소 구조:
        {
            "horse_id": int | str,
            "is_first_run": bool,   # 첫 출전 여부 (True/False)
            "weight": float | int | None | 0
        }

    LOGIC
    -----
    1. weight가 존재하면 (weight != 0 and weight is not None)
       → 그대로 반환

    2. weight가 0 또는 None이면 결측으로 간주

    3. 결측 처리 규칙:
        3-1. 해당 말이 여러 번 출전한 경우
             → 해당 horse_id의 평균 weight로 대체

        3-2. 해당 말이 첫 출전인 경우
             → 전체 데이터의 평균 weight로 대체

    OUTPUT
    ------
    list[dict]
        {
            "horse_id": int | str,
            "weight": float
        }
        (보정된 weight 반환)
    """
    pass

In [ ]:
def fill_weight(data: list[dict]) -> list[dict]:
    """
    경주마 몸무게 결측치 보정 함수
    """

    # 1) 전체 평균 계산용
    total_sum = 0
    total_cnt = 0

    # 2) 말(hrno)별 평균 계산용
    horse_sum = {}
    horse_cnt = {}

    # -------------------------
    # 1차 패스: 통계 계산
    # -------------------------
    for row in data:
        hrno = row["hrno"]
        w = row.get("wgHr")

        if w is None or w == 0:
            continue

        total_sum += w
        total_cnt += 1

        if hrno not in horse_sum:
            horse_sum[hrno] = 0
            horse_cnt[hrno] = 0

        horse_sum[hrno] += w
        horse_cnt[hrno] += 1

    # 평균 계산
    global_mean = total_sum / total_cnt if total_cnt > 0 else 0

    horse_mean = {}
    for h in horse_sum:
        horse_mean[h] = horse_sum[h] / horse_cnt[h]

    # -------------------------
    # 2차 패스: 결측치 채우기
    # -------------------------
    result = []

    for row in data:
        hrno = row["hrno"]
        w = row.get("wgHr")
        is_first = row.get("is_first_run", False)

        # 정상값 유지
        if w is not None and w != 0:
            result.append({
                "hrno": hrno,
                "wgHr": float(w)
            })
            continue

        # 결측 처리
        if is_first:
            fill_value = global_mean
        else:
            fill_value = horse_mean.get(hrno, global_mean)

        result.append({
            "hrno": hrno,
            "wgHr": float(fill_value)
        })

    return result

In [ ]:
import numpy as np
import pandas as pd
#말의 몸무게 


# 말의 몸무게 = IF  (말이 여러번 출전했다면) (해당 말의 시즌 몸무게 평균) ELSE (말이 첫 출전이면) (모든 말의 평균 몸무게)
# 함수의 아웃풋은 '말의 몸무게' 값 1개만

"""
말의 몸무게의 결측치는 0인 경우다
---그러니까 0인경울 NaN한다
--- 말의 몸무게가 0인 경우
말이 여러번 출전했다면 그 해당 마번의 평균 몸무게를 fillna
말이 첫출전이라면 전체 평균 말의 몸무게
"""

def fill_wghr(df, col='wgHr', group_col='hrno'):
    df = df.copy()

    # 1. 0 → NaN 처리
    df[col] = df[col].replace(0, np.nan)

    # 2. float 변환 (중요: mean 때문에)
    df[col] = df[col].astype(float)

    # 3. hrno 기준 평균으로 채우기
    df[col] = df[col].fillna(df.groupby(group_col)[col].transform('mean'))

    # 4. 그래도 남으면 전체 평균
    df[col] = df[col].fillna(df[col].mean())

    return df

df = fill_wghr(df)

np.int64(0)

In [66]:
df.columns

Index(['chaksun', 'diffTot', 'divide', 'hrName', 'hrno', 'jkName', 'jkNo',
       'meet', 'noracefl', 'prow', 'prowName', 'prtr', 'prtrName', 'rankKind',
       'rc10dusu', 'rcAge', 'rcBudam', 'rcChul', 'rcCode', 'rcDate', 'rcDiff2',
       'rcDiff3', 'rcDiff4', 'rcDiff5', 'rcDist', 'rcFrflag', 'rcGrade',
       'rcHrfor', 'rcHrnew', 'rcNo', 'rcNrace', 'rcOrd', 'rcP1Odd', 'rcP1Sale',
       'rcP2Odd', 'rcP2Sale', 'rcP3Odd', 'rcP3Sale', 'rcP4Odd', 'rcP4Sale',
       'rcP5Odd', 'rcP5Sale', 'rcP6Odd', 'rcP6Sale', 'rcP8Odd', 'rcP8Sale',
       'rcPlansu', 'rcRank', 'rcSex', 'rcSpcbu', 'rcTime', 'rcVtdusu',
       'rundayth', 'track', 'weath', 'wgHr'],
      dtype='str')

In [88]:
df['prtr'].info


<bound method Series.info of 0        70160.0
1        70148.0
2        70200.0
3        70170.0
4        70174.0
          ...   
11015    70177.0
11016    70148.0
11017    70244.0
11018    70160.0
11019    70096.0
Name: prtr, Length: 11020, dtype: float64>

In [89]:
df["prtr"].isnull().sum()

np.int64(4)